# Class 5 - Control Flow & Loops
### Data Analytics in Python | Week 2, Saturday

Until now, your code has run top to bottom, every line, every time. That's not
how real programs work.

Control flow is how you make decisions (if a patient has a fever or if a transaction is fraudulent, flag it) and
repeat work (process every row in a dataset). It's the engine behind every
analytics pipeline you'll ever build.

**One key insight before we start:** the list comprehension `[expr for x in items]`
that you learned in Class 3 is literally a `for` loop under the hood. Today you
learn to write that loop explicitly - and unlock everything it can do that
comprehensions can't: early exit, accumulators, stateful operations, nested logic.

## 1. Choosing the Right Loop

| Situation | Best Choice |
|---|---|
| Process **`every item`** in a collection | `for` loop |
| Repeat actions indefinitely **`until a condition changes`** | `while` loop |
| Need **`both index and value`** | `enumerate()` |
| To parse multiple parallel arrays/lists together, item by item. | `zip()` |
| Generate a **`sequence of numbers`** or or run a loop a fixed number of times. | `range()` |
| Stop immediately after finding a value | `break` |
| Skip invalid records (anomalies or missing data blocks) and immediately jump to the next item. | `continue` |

**Today:**
- `if` / `elif` / `else` - decision trees
- `for` loops - iterating over any iterable
- `range()`, `enumerate()`, `zip()` - Pythonic iteration tools
- `while` loops - iteration by condition
- `break`, `continue`, `pass` - loop control
- Nested loops - matrices, coordinate grids, and 2D arrays
- Practical: Grade classification pipeline

## **1. if / elif / else**

Conditional execution. Python evaluates each condition top to bottom and executes the **first** matching block. Only one branch runs.

### 1a. Basic structure and Multiple conditions using Logical operators

In [1]:
# Basic structure
temp = 38.9   # patient temperature

if temp >= 39.5:
    print("HIGH FEVER - immediate attention required")
elif temp >= 38.0:
    print("FEVER - monitor closely")
elif temp >= 37.5:
    print("Mild elevation - recheck in 2 hours")
else:
    print("Normal temperature")

# Multiple conditions using logical operators
temp, bp = 38.9, 145

if temp >= 38.0 and bp > 140:
    print("\nDual alert: fever + hypertension")
elif temp >= 38.0 or bp > 140:
    print("\nSingle alert: one marker elevated")
else:
    print("\nAll vitals normal")

FEVER - monitor closely

Dual alert: fever + hypertension


In [3]:
temp, bp = 120,110; print(temp, bp)

120 110


### 1b. Short-circuit evaluation matters in conditions

In [ ]:
# Short-circuit evaluation matters in conditions
# Python stops evaluating as soon as the result is known

def expensive_check(name):
    # Simulates a slow database query - takes time in production
    print(f"  [querying database for {name}...]")
    return True

patient_id = "P0042"
is_valid_format = len(patient_id) == 5 and patient_id.startswith("P")

print(f"ID format valid: {is_valid_format}")

#due to short-circuit, the database is only queried if the format is valid
if is_valid_format and expensive_check(patient_id):
    print("Patient record found")

#if format is invalid, expensive_check is never called
if not is_valid_format and expensive_check("P???"):
    print("This won't run")
print("(expensive_check was NOT called above - short-circuit)")

ID format valid: True
  [querying database for P0042...]
Patient record found
(expensive_check was NOT called above - short-circuit)


### 1c. Nested conditions (inside a Function)

In [ ]:
# Nested conditions - patient triage scoring
def triage_score(temp, bp_sys, glucose, age):
    score = 0
    # Temperature component
    if temp >= 39.5:
        score += 3
    elif temp >= 38.0:
        score += 2
    elif temp >= 37.5:
        score += 1
    # Blood pressure component
    if bp_sys >= 180:
        score += 3
    elif bp_sys >= 140:
        score += 2
    elif bp_sys >= 130:
        score += 1
    # Glucose component
    if glucose >= 12.0:
        score += 2
    elif glucose >= 7.0:
        score += 1
    # Age risk
    if age >= 65:
        score += 1
    return score

def triage_level(score):
    if score >= 7:
        return "CRITICAL"
    elif score >= 4:
        return "HIGH"
    elif score >= 2:
        return "MEDIUM"
    else:
        return "LOW"

#test cases
patients = [
    ("Ahmad", 38.9, 145, 9.4, 45),
    ("Sara",  36.7, 118, 5.1, 28),
    ("Bilal", 39.6, 175, 12.3, 67),
]
print(f"{'Name':<10} {'Score':>6} {'Level'}")
print("-" * 28)
for name, temp, bp, glu, age in patients:
    sc = triage_score(temp, bp, glu, age)
    print(f"{name:<10} {sc:>6}   {triage_level(sc)}")

Name        Score Level
----------------------------
Ahmad           5   HIGH
Sara            0   LOW
Bilal           8   CRITICAL


## **2. for loops**

Iterate over **any sequence:** `string`, `list`, `tuple`, `dict`, `set`, `range`, or `file`. Python's `for` loop is clean and readable - no index management unless you need it.

### 2a. For loop on String, List, and Dictionary

In [4]:
# Iterate over different types
print("=== Over a string ===")
for char in "ATCG":
    print(f"  {char}", end=" ")
print()

print("\n=== Over a list ===")
scores = [88, 55, 94, 43, 77]
for score in scores:
    print(f"  {score}", end=" ")
print()

print("\n=== Over a dict (iterates keys by default) ===")
vitals = {"temp": 38.7, "bp": 145, "hr": 92}
for key in vitals:
    print(f"  {key}: {vitals[key]}")

print("\n=== Over dict .items() (key-value pairs) ===")
for key, val in vitals.items():
    print(f"  {key:<6} = {val}")

=== Over a string ===
  A   T   C   G 

=== Over a list ===
  88   55   94   43   77 

=== Over a dict (iterates keys by default) ===
  temp: 38.7
  bp: 145
  hr: 92

=== Over dict .items() (key-value pairs) ===
  temp   = 38.7
  bp     = 145
  hr     = 92


### 2b. Building `Patterns using` `for loop`

In [5]:
# Building results with a for loop - the accumulator pattern
scores = [88, 55, 94, 43, 77, 91, 60, 82, 38, 96]

# Pattern 1: accumulate a running total
total = 0
for s in scores:
    total += s
mean = total / len(scores)
print(f"Mean: {mean:.2f}")

# Pattern 2: build a filtered list
passing = []
for s in scores:
    if s >= 60:
        passing.append(s)
print(f"Passing (>=60): {passing}")
# Note: this is what list comprehension does: [s for s in scores if s >= 60]

# Pattern 3: count occurrences by category
grade_counts = {"A": 0, "B": 0, "C": 0, "D": 0, "F": 0}
for s in scores:
    if s >= 90:
        grade_counts["A"] += 1
    elif s >= 80:
        grade_counts["B"] += 1
    elif s >= 70:
        grade_counts["C"] += 1
    elif s >= 60:
        grade_counts["D"] += 1
    else:
        grade_counts["F"] += 1

print(f"Grade distribution: {grade_counts}")

Mean: 72.40
Passing (>=60): [88, 94, 77, 91, 60, 82, 96]
Grade distribution: {'A': 3, 'B': 2, 'C': 1, 'D': 1, 'F': 3}


## **3. range(), enumerate(), zip()**

Three built-in generators that power most for-loop patterns in Python.

### 3a. range()

In [6]:
# range(start, stop, step) - generates integers on demand
print("=== range() ===")
print(f"range(5):      {list(range(5))}")       # 0,1,2,3,4
print(f"range(2,8):    {list(range(2,8))}")      # 2,3,4,5,6,7
print(f"range(0,20,3): {list(range(0,20,3))}")   # 0,3,6,9,12,15,18
print(f"range(10,0,-2):{list(range(10,0,-2))}")  # 10,8,6,4,2

# Generate a sequence of patient IDs
patient_ids = [f"P{i:04d}" for i in range(1, 6)]
print(f"\nPatient IDs: {patient_ids}")

=== range() ===
range(5):      [0, 1, 2, 3, 4]
range(2,8):    [2, 3, 4, 5, 6, 7]
range(0,20,3): [0, 3, 6, 9, 12, 15, 18]
range(10,0,-2):[10, 8, 6, 4, 2]

Patient IDs: ['P0001', 'P0002', 'P0003', 'P0004', 'P0005']


### 3b. enumerate() - returns both index and value

In [ ]:
# enumerate() - index + value together (NEVER use range(len(x)) manually)
genes = ["BRCA1", "TP53", "KRAS", "EGFR"]

print("=== enumerate() ===")
##the manual (bad) way:
# for i in range(len(genes)):
#     print(i, genes[i])

#the right way:
for i, gene in enumerate(genes):
    print(f"  Gene {i}: {gene}")

print()
for i, gene in enumerate(genes, start=1):   # start index at 1
    print(f"  Rank {i}: {gene}")

#practical: track which scores are failing AND their position
scores = [88, 55, 94, 43, 77, 91, 60]
print("\nFailing positions:")
for i, s in enumerate(scores):
    if s < 60:
        print(f"  Index {i}: score {s}")

=== enumerate() ===
  Gene 0: BRCA1
  Gene 1: TP53
  Gene 2: KRAS
  Gene 3: EGFR

  Rank 1: BRCA1
  Rank 2: TP53
  Rank 3: KRAS
  Rank 4: EGFR

Failing positions:
  Index 1: score 55
  Index 3: score 43


#### can we set custom index in enumerate?!!

In [5]:
retail_locations = ["Warehouse_A", "Storefront_East", "Hub_North", "Outlet_South"]

print("=== To Track Index and Values ===")
for index, site in enumerate(retail_locations, start=1):
    print(f"  Priority Rank {index}: Processing operational logs for -> {site}")

=== To Track Index and Values ===
  Priority Rank 1: Processing operational logs for -> Warehouse_A
  Priority Rank 2: Processing operational logs for -> Storefront_East
  Priority Rank 3: Processing operational logs for -> Hub_North
  Priority Rank 4: Processing operational logs for -> Outlet_South


### 3c. zip() - pair up two (or more) iterables element by element


In [2]:
names  = ["Ahmad", "Sara", "Bilal", "Zara"]
scores = [88, 92, 74, 61]
grades = ["B", "A", "C", "D"]
ab = list(zip(names, scores, grades))

print(f'ab is a List of tuples: {ab}')

print("=== zip() ===")
for name, score, grade in zip(names, scores, grades):
    print(f"  {name:<10} {score:>5}   {grade}")

ab is a List of tuples: [('Ahmad', 88, 'B'), ('Sara', 92, 'A'), ('Bilal', 74, 'C'), ('Zara', 61, 'D')]
=== zip() ===
  Ahmad         88   B
  Sara          92   A
  Bilal         74   C
  Zara          61   D


In [4]:
# zip stops at the SHORTEST iterable - important to know
a = [1, 2, 3, 4, 5]
b = ["x", "y", "z"]
print(f"\nzip([1..5], ['x','y','z']): {list(zip(a, b))}")  # only 3 pairs


zip([1..5], ['x','y','z']): [(1, 'x'), (2, 'y'), (3, 'z')]


In [3]:
# Practical: compare two patient readings
baseline = [36.5, 120, 75, 5.0]
current  = [38.9, 145, 92, 9.4]
markers  = ["temp", "bp_sys", "hr", "glucose"]

print("\nPatient delta:")
for marker, base, curr in zip(markers, baseline, current):
    delta = curr - base
    flag  = " <--- ELEVATED" if delta > 0 else ""
    print(f"  {marker:<10} {base:>6.1f} ---> {curr:>5.1f}  ({delta:+.1f}){flag}")


Patient delta:
  temp         36.5 --->  38.9  (+2.4) <--- ELEVATED
  bp_sys      120.0 ---> 145.0  (+25.0) <--- ELEVATED
  hr           75.0 --->  92.0  (+17.0) <--- ELEVATED
  glucose       5.0 --->   9.4  (+4.4) <--- ELEVATED


## **4. while loops**

A `for` loop runs a fixed number of iterations. A `while` loop runs **indefinitely as long as a condition is true**. `Use it` when you don't know in advance how many iterations you'll need.

### **The Danger of Infinite Loops**
An infinite loop occurs when the target exit condition never resolves to `False`. For example, running the following snippet will print continuously because no modification step is applied to counter `x`:
```python
x = 0
while x < 10:
    print(x)  # This loops forever because x is never incremented!
```

### 4a. Basic while loop

In [6]:
count = 0
while count < 5:
    print(f"  count = {count}")
    count += 1

print(f"Loop ended, count = {count}")

#classic use: retry until success (simulated)
import random
random.seed(42)

attempts = 0
while True:    # infinite loop - need an explicit break
    reading = random.uniform(35, 41)
    attempts += 1
    if 36.0 <= reading <= 37.5:   # valid range
        break                      # exit the loop

print(f"\nValid reading {reading:.2f}°C found after {attempts} attempts")

  count = 0
  count = 1
  count = 2
  count = 3
  count = 4
Loop ended, count = 5

Valid reading 36.65°C found after 3 attempts


### 4b. while with multiple conditions

In [ ]:
# Simulate a data quality loop: process records until we've found 4 valid ones
all_records = [
    {"id": "P01", "temp": 38.9, "valid": True},
    {"id": "P02", "temp": None, "valid": False},    # missing value
    {"id": "P03", "temp": 37.2, "valid": True},
    {"id": "P04", "temp": -99,  "valid": False},    # sentinel value
    {"id": "P05", "temp": 36.8, "valid": True},
    {"id": "P06", "temp": 39.1, "valid": True},
    {"id": "P07", "temp": 37.5, "valid": True},
    {"id": "P08", "temp": None, "valid": False},
]

valid_records = []
idx = 0
while idx < len(all_records) and len(valid_records) < 4:
    rec = all_records[idx]
    if rec["valid"]:
        valid_records.append(rec)
    idx += 1

print(f'Valid records: {valid_records}\n and its type is: {type(valid_records)}')
print()

print(f"Processed {idx} records, found {len(valid_records)} valid:")
for r in valid_records:
    print(f"  {r['id']}  temp={r['temp']}")

Valid records: [{'id': 'P01', 'temp': 38.9, 'valid': True}, {'id': 'P03', 'temp': 37.2, 'valid': True}, {'id': 'P05', 'temp': 36.8, 'valid': True}, {'id': 'P06', 'temp': 39.1, 'valid': True}]
 and its type is: <class 'list'>

Processed 6 records, found 4 valid:
  P01  temp=38.9
  P03  temp=37.2
  P05  temp=36.8
  P06  temp=39.1


## **5. break, continue, pass**

Three loop control keywords that give you fine-grained control.

### 5a. break - exit the loop immediately

In [11]:
print("=== break ===")
sequences = ["ATCGATCG", "GCTAGCTA", "NNNATCG", "TTTAAA"]

for seq in sequences:
    if "NNN" in seq:
        print(f"  Bad sequence found: {seq} - stopping batch")
        break
    print(f"  Processing: {seq}")

=== break ===
  Processing: ATCGATCG
  Processing: GCTAGCTA
  Bad sequence found: NNNATCG - stopping batch


### 5b. continue - skip this iteration, continue to next

In [12]:
print("\n=== continue ===")
readings = [37.2, None, 38.1, None, 36.9, 39.0]

valid_sum = 0
valid_count = 0
for r in readings:
    if r is None:
        continue               # skip missing values
    valid_sum += r
    valid_count += 1

print(f"  Valid readings: {valid_count}/{len(readings)}")
print(f"  Mean of valid:  {valid_sum/valid_count:.2f}°C")


=== continue ===
  Valid readings: 4/6
  Mean of valid:  37.80°C


### 5c. pass - do nothing (placeholder, or intentional empty block)

In [13]:
print("\n=== pass ===")
data = {"A": 10, "B": 0, "C": 15}
for key, val in data.items():
    if val == 0:
        pass    # handle zero case later - for now, skip silently
    else:
        print(f"  {key}: ratio = {val/25:.2f}")


=== pass ===
  A: ratio = 0.40
  C: ratio = 0.60


### 5d. The for...else construct - `runs only if loop completed WITHOUT a break`

In [14]:
# Less known but genuinely useful for search tasks

def find_gene(gene_list, target):
    for i, gene in enumerate(gene_list):
        if gene == target:
            print(f"  Found '{target}' at index {i}")
            break
    else:
        # This runs only if break was never triggered
        print(f"  '{target}' not found in list")

gene_panel = ["BRCA1", "TP53", "KRAS", "EGFR"]

find_gene(gene_panel, "KRAS")    # Found
find_gene(gene_panel, "MYC")     # Not found

  Found 'KRAS' at index 2
  'MYC' not found in list


## **6. Nested loops**

In [20]:
# 2D data - iterating over a matrix (gene expression)
gene_expr = [
    [1.23, 0.87, 2.14],   # Patient 1
    [0.91, 1.44, 0.78],   # Patient 2
    [2.05, 1.89, 1.33],   # Patient 3
]
genes    = ["BRCA1", "TP53", "KRAS"]
patients = ["P1", "P2", "P3"]

print("Gene expression (row=patient, col=gene):")
print(f"{'':>5} " + "  ".join(f"{g:>8}" for g in genes))
print("-" * 35)
for i, row in enumerate(gene_expr):
    values = "  ".join(f"{v:>8.2f}" for v in row)
    print(f"{patients[i]:>5} {values}")

print()

print(f'nested list:\n {gene_expr}')

#find all overexpressed genes (value > 1.5) with their coordinates
print("\nOverexpressed (value > 1.5):")
for p_idx, row in enumerate(gene_expr): #loops over external list (nested list - so two for loops)
    for g_idx, val in enumerate(row):    #row is internal list, 2nd loop for that list
        if val > 1.5:
            print(f"  {patients[p_idx]}, {genes[g_idx]}: {val:.2f}")

Gene expression (row=patient, col=gene):
         BRCA1      TP53      KRAS
-----------------------------------
   P1     1.23      0.87      2.14
   P2     0.91      1.44      0.78
   P3     2.05      1.89      1.33

nested list:
 [[1.23, 0.87, 2.14], [0.91, 1.44, 0.78], [2.05, 1.89, 1.33]]

Overexpressed (value > 1.5):
  P1, KRAS: 2.14
  P3, BRCA1: 2.05
  P3, TP53: 1.89


### Generating combinations - codon enumeration
You have four choices `'A', 'T', 'G', 'C'`, you have to make all possible combinations of these but you can `use only three of these at a time` also elements can repeat. This is a `repeating permutation problem`.

combination = choice_1 + choice_2 + choice_3

A combination has three choices/elements at a time so `three for loops`.
 when you pick say 'T' then you still have 4 elements to choose from as `elements can repeat`.

In [ ]:
bases = ["A", "T", "C", "G"]
stop_codons = {"TAA", "TAG", "TGA"}

print("All start-codon (ATG) and stop-codon combinations:")
print("Start codons: ATG")
print(f"Stop codons: {stop_codons}")
print()

# All possible codons (4^3 = 64)
total_codons = 0
coding_codons = []
for b1 in bases:
    for b2 in bases:
        for b3 in bases:
            codon = b1 + b2 + b3
            total_codons += 1
            if codon not in stop_codons:
                coding_codons.append(codon)

print(f"Total codons: {total_codons}")
print(f"Coding codons (non-stop): {len(coding_codons)}")
print(f"Stop codons: {total_codons - len(coding_codons)}")
print(f"First 12: {coding_codons[:12]}")

All start-codon (ATG) and stop-codon combinations:
Start codons: ATG
Stop codons: {'TAG', 'TGA', 'TAA'}

Total codons: 64
Coding codons (non-stop): 61
Stop codons: 3
First 12: ['AAA', 'AAT', 'AAC', 'AAG', 'ATA', 'ATT', 'ATC', 'ATG', 'ACA', 'ACT', 'ACC', 'ACG']


## **Practical**

### Practical 1 - Loop variable persists after loop ends
**Context:** A student processes a list of patients and then tries to use the loop variable to report the last patient processed. What happens?

In [15]:
# Predict: what is 'gene' after the loop ends?
genes = ["BRCA1", "TP53", "KRAS"]

for gene in genes:
    pass   # just iterate - don't do anything

print(f"After loop, gene = {gene!r}")    # "KRAS" - last value!

# This is a real bug waiting to happen:
records = [{"id": "P01", "valid": True},
           {"id": "P02", "valid": False},
           {"id": "P03", "valid": True}]

for rec in records:
    if rec["valid"]:
        last_valid = rec   # track inside loop
    # 'rec' still exists and equals the LAST record - P03

print(f"\nrec after loop:        {rec}")         # P03 (last iterated)
print(f"last_valid after loop: {last_valid}")    # P03 (last valid)

# Lesson: if you want the LAST valid record, track it explicitly with a variable.
# Never assume 'rec' holds your target after the loop.

After loop, gene = 'KRAS'

rec after loop:        {'id': 'P03', 'valid': True}
last_valid after loop: {'id': 'P03', 'valid': True}


### Practical 2 - enumerate() vs manual indexing: what breaks first?
**Context:** A student needs to display a numbered list of results. They write two versions.

In [16]:
results = ["Sequence loaded", "Quality check: PASS", "Alignment: 94.3%",
           "Variant calling: 142 SNPs", "Report generated"]

print("=== Manual indexing (bad habit) ===")
i = 0
for r in results:
    print(f"  Step {i+1}: {r}")
    i += 1

print("\n=== enumerate() (correct) ===")
for i, r in enumerate(results, start=1):
    print(f"  Step {i}: {r}")

# The real difference: what happens if you accidentally skip the counter?
print("\n=== What breaks with manual i ===")
j = 0
for r in results:
    if "PASS" in r:
        continue      # skip - but j is NOT incremented here!
    print(f"  Step {j+1}: {r}")
    j += 1

print("\n=== enumerate() is immune to this ===")
for i, r in enumerate(results, start=1):
    if "PASS" in r:
        continue
    print(f"  Step {i}: {r}")    # i is always the TRUE position

=== Manual indexing (bad habit) ===
  Step 1: Sequence loaded
  Step 2: Quality check: PASS
  Step 3: Alignment: 94.3%
  Step 4: Variant calling: 142 SNPs
  Step 5: Report generated

=== enumerate() (correct) ===
  Step 1: Sequence loaded
  Step 2: Quality check: PASS
  Step 3: Alignment: 94.3%
  Step 4: Variant calling: 142 SNPs
  Step 5: Report generated

=== What breaks with manual i ===
  Step 1: Sequence loaded
  Step 2: Alignment: 94.3%
  Step 3: Variant calling: 142 SNPs
  Step 4: Report generated

=== enumerate() is immune to this ===
  Step 1: Sequence loaded
  Step 3: Alignment: 94.3%
  Step 4: Variant calling: 142 SNPs
  Step 5: Report generated


### Practical 3 - break in a search vs scanning the whole list
**Context:** A genome has 3 million base pairs. You're looking for the first occurrence of a restriction enzyme site 'GAATTC'. Should you scan everything?

In [18]:
import time, random, string

# Simulate a large genomic sequence
random.seed(42)
genome = "".join(random.choices("ATCG", k=3_000_000))
# Plant our target at position 50,000
genome = genome[:50_000] + "GAATTC" + genome[50_006:]

target = "GAATTC"

# Method 1: scan everything (no break)
t0 = time.time()
found_pos_full = -1
for i in range(len(genome) - len(target) + 1):
    if genome[i:i+len(target)] == target:
        found_pos_full = i
        # No break - keeps going!
t_full = time.time() - t0

# Method 2: stop as soon as found
t0 = time.time()
found_pos_break = -1
for i in range(len(genome) - len(target) + 1):
    if genome[i:i+len(target)] == target:
        found_pos_break = i
        break              # exit immediately
t_break = time.time() - t0

print(f"Target at position: {found_pos_break}")
print(f"Without break: {t_full:.4f}s  (scanned all 3M positions)")
print(f"With break:    {t_break:.4f}s  (stopped at position 50,000)")
print(f"Speedup: {t_full/(t_break+1e-9):.0f}x faster")
print("\nLesson: break isn't just style - it's a correctness AND performance tool")

Target at position: 2664
Without break: 1.2970s  (scanned all 3M positions)
With break:    0.0018s  (stopped at position 50,000)
Speedup: 741x faster

Lesson: break isn't just style - it's a correctness AND performance tool


### Practical 4 - zip() mismatch trap
**Context:** A data entry team submits two lists: patient IDs and their test results. The lists have different lengths. What does zip() do? What should it do?

In [19]:
patient_ids  = ["P001", "P002", "P003", "P004", "P005"]
test_results = [8.2, 5.1, 11.3, 9.4]   # ONLY 4 results - data entry error!

print("=== Default zip() behaviour ===")
pairs = list(zip(patient_ids, test_results))
print(f"Pairs: {pairs}")
print(f"Expected: {len(patient_ids)} pairs")
print(f"Got:      {len(pairs)} pairs  ← P005 silently dropped!")

# How to detect the mismatch BEFORE it silently loses data
print("\n=== Safe practice: check lengths first ===")
if len(patient_ids) != len(test_results):
    print(f"WARNING: length mismatch!")
    print(f"  IDs: {len(patient_ids)}, Results: {len(test_results)}")
    missing_ids = patient_ids[len(test_results):]
    print(f"  Missing results for: {missing_ids}")
else:
    for pid, result in zip(patient_ids, test_results):
        print(f"  {pid}: {result}")

# Python 3.10+ has zip(strict=True) which raises ValueError on mismatch
# zip(patient_ids, test_results, strict=True)  ← uncomment if Python 3.10+

=== Default zip() behaviour ===
Pairs: [('P001', 8.2), ('P002', 5.1), ('P003', 11.3), ('P004', 9.4)]
Expected: 5 pairs
Got:      4 pairs  ← P005 silently dropped!

=== Safe practice: check lengths first ===
  IDs: 5, Results: 4
  Missing results for: ['P005']


## 8. Practical Exercise - E-Commerce Sales Pipeline
**Difficulty: Medium**

In [ ]:
#Analytics Dataset: 20 Customer profiles tracking transaction data logs
customer_profiles = [
    {"name": "Ahmad Raza",     "purchases": [88, 92, 74, 85, 90]},
    {"name": "Sara Khan",      "purchases": [92, 88, 95, 91, 89]},
    {"name": "Bilal Ahmed",    "purchases": [74, 61, 78, 55, 70]},
    {"name": "Zara Malik",     "purchases": [61, 55, 43, 70, 65]},
    {"name": "Hassan Ali",     "purchases": [95, 98, 91, 94, 97]},
    {"name": "Nida Butt",      "purchases": [83, 79, 87, 82, 88]},
    {"name": "Usman Sheikh",   "purchases": [47, 55, 38, 62, 50]},
    {"name": "Ayesha Nawaz",   "purchases": [78, 82, 75, 80, 77]},
    {"name": "Faisal Iqbal",   "purchases": [89, 91, 86, 92, 88]},
    {"name": "Mariam Syed",    "purchases": [56, 48, 61, 52, 58]},
    {"name": "Tariq Mahmood",  "purchases": [91, 94, 89, 96, 92]},
    {"name": "Sana Mirza",     "purchases": [67, 72, 65, 70, 68]},
    {"name": "Ali Hassan",     "purchases": [73, 68, 79, 74, 71]},
    {"name": "Hina Qureshi",   "purchases": [84, 88, 81, 86, 83]},
    {"name": "Kamran Rauf",    "purchases": [38, 42, 35, 48, 40]},
    {"name": "Fatima Javed",   "purchases": [79, 83, 77, 81, 76]},
    {"name": "Omar Farooq",    "purchases": [93, 90, 95, 88, 94]},
    {"name": "Rabia Shahid",   "purchases": [65, 69, 63, 71, 67]},
    {"name": "Imran Khalid",   "purchases": [82, 85, 79, 87, 83]},
    {"name": "Mehwish Aslam",  "purchases": [71, 74, 68, 76, 72]},
]
DEPARTMENTS = ["Electronics", "Clothing", "Home_Goods", "Beauty", "Groceries"]

# Compute ticket value summaries across customer records
for profile in customer_profiles:
    calculated_avg = sum(profile["purchases"]) / len(profile["purchases"])
    profile["avg_spend"] = round(calculated_avg, 2)

    if calculated_avg >= 90:   profile["segment"] = "Platinum VIP"
    elif calculated_avg >= 80: profile["segment"] = "Gold Preferred"
    elif calculated_avg >= 70: profile["segment"] = "Silver Frequent"
    elif calculated_avg >= 60: profile["segment"] = "Standard Retail"
    else:                      profile["segment"] = "Low Engagement"

print(f"{'Customer Name':<20} {'Avg Spend':>10} {'Account Segment':>18}")
print("-" * 52)
for profile in sorted(customer_profiles, key=lambda item: -item["avg_spend"]):
    star_flag = " *" if profile["segment"] == "Platinum VIP" else ""
    print(f"{profile['name']:<20} ${profile['avg_spend']:>9.2f}   {profile['segment']:>18}{star_flag}")

## 9. Practical Exercise - Grade Classification Pipeline
**Difficulty: Medium**

In [21]:
# Student dataset - 20 students, 5 subject scores each
students = [
    {"name": "Ahmad Raza",     "scores": [88, 92, 74, 85, 90]},
    {"name": "Sara Khan",      "scores": [92, 88, 95, 91, 89]},
    {"name": "Bilal Ahmed",    "scores": [74, 61, 78, 55, 70]},
    {"name": "Zara Malik",     "scores": [61, 55, 43, 70, 65]},
    {"name": "Hassan Ali",     "scores": [95, 98, 91, 94, 97]},
    {"name": "Nida Butt",      "scores": [83, 79, 87, 82, 88]},
    {"name": "Usman Sheikh",   "scores": [47, 55, 38, 62, 50]},
    {"name": "Ayesha Nawaz",   "scores": [78, 82, 75, 80, 77]},
    {"name": "Faisal Iqbal",   "scores": [89, 91, 86, 92, 88]},
    {"name": "Mariam Syed",    "scores": [56, 48, 61, 52, 58]},
    {"name": "Tariq Mahmood",  "scores": [91, 94, 89, 96, 92]},
    {"name": "Sana Mirza",     "scores": [67, 72, 65, 70, 68]},
    {"name": "Ali Hassan",     "scores": [73, 68, 79, 74, 71]},
    {"name": "Hina Qureshi",   "scores": [84, 88, 81, 86, 83]},
    {"name": "Kamran Rauf",    "scores": [38, 42, 35, 48, 40]},
    {"name": "Fatima Javed",   "scores": [79, 83, 77, 81, 76]},
    {"name": "Omar Farooq",    "scores": [93, 90, 95, 88, 94]},
    {"name": "Rabia Shahid",   "scores": [65, 69, 63, 71, 67]},
    {"name": "Imran Khalid",   "scores": [82, 85, 79, 87, 83]},
    {"name": "Mehwish Aslam",  "scores": [71, 74, 68, 76, 72]},
]
SUBJECTS = ["Math", "Python", "Stats", "DS", "Comm"]

# Compute average for each student (add 'avg' and 'grade' to dict)
for student in students:
    avg = sum(student["scores"]) / len(student["scores"])
    student["avg"] = round(avg, 2)

    if avg >= 90:   student["grade"] = "A"
    elif avg >= 80: student["grade"] = "B"
    elif avg >= 70: student["grade"] = "C"
    elif avg >= 60: student["grade"] = "D"
    else:           student["grade"] = "F"

print(f"{'Name':<20} {'Avg':>6} {'Grade':>6}  {'Status':>8}")
print("-" * 46)
for s in sorted(students, key=lambda x: -x["avg"]):
    status = "PASS" if s["grade"] != "F" else "FAIL"
    flag   = " *" if s["avg"] >= 90 else ""
    print(f"{s['name']:<20} {s['avg']:>6.2f}   {s['grade']:>4}   {status:>7}{flag}")

Name                    Avg  Grade    Status
----------------------------------------------
Hassan Ali            95.00      A      PASS *
Tariq Mahmood         92.40      A      PASS *
Omar Farooq           92.00      A      PASS *
Sara Khan             91.00      A      PASS *
Faisal Iqbal          89.20      B      PASS
Ahmad Raza            85.80      B      PASS
Hina Qureshi          84.40      B      PASS
Nida Butt             83.80      B      PASS
Imran Khalid          83.20      B      PASS
Fatima Javed          79.20      C      PASS
Ayesha Nawaz          78.40      C      PASS
Ali Hassan            73.00      C      PASS
Mehwish Aslam         72.20      C      PASS
Sana Mirza            68.40      D      PASS
Bilal Ahmed           67.60      D      PASS
Rabia Shahid          67.00      D      PASS
Zara Malik            58.80      F      FAIL
Mariam Syed           55.00      F      FAIL
Usman Sheikh          50.40      F      FAIL
Kamran Rauf           40.60      F      FAIL


In [21]:
# Grade distribution
grade_counts = {}
for s in students:
    g = s["grade"]
    grade_counts[g] = grade_counts.get(g, 0) + 1

print("Grade distribution:")
for grade in "ABCDF":
    count = grade_counts.get(grade, 0)
    bar   = "█" * count
    print(f"  {grade}: {bar:<20} {count}")

# Subject analysis - find subjects with class average below 70
subject_totals = [0] * len(SUBJECTS)
for s in students:
    for j, score in enumerate(s["scores"]):
        subject_totals[j] += score

subject_avgs = [total / len(students) for total in subject_totals]
print("\nSubject averages:")
flagged = []
for i, (subj, avg) in enumerate(zip(SUBJECTS, subject_avgs)):
    flag = " ← below 70, review curriculum" if avg < 70 else ""
    print(f"  {subj:<8} {avg:.1f}{flag}")
    if avg < 70:
        flagged.append(subj)

if flagged:
    print(f"\nSubjects requiring curriculum review: {flagged}")

Grade distribution:
  A: ████                 4
  B: █████                5
  C: ████                 4
  D: ███                  3
  F: ████                 4

Subject averages:
  Math     75.3
  Python   75.7
  Stats    73.0
  DS       77.0
  Comm     75.9


In [22]:
# Top 3 and bottom 3 - sort by average
ranked = sorted(students, key=lambda s: s["avg"], reverse=True)

print("TOP 3 STUDENTS")
for i, s in enumerate(ranked[:3], 1):
    print(f"  {i}. {s['name']:<20} avg={s['avg']:.2f}  grade={s['grade']}")

print("\nBOTTOM 3 STUDENTS")
for i, s in enumerate(reversed(ranked[-3:]), 1):
    print(f"  {i}. {s['name']:<20} avg={s['avg']:.2f}  grade={s['grade']}")

# Count pass/fail
passing = sum(1 for s in students if s["grade"] != "F")
failing = len(students) - passing
print(f"\nPass: {passing}/{len(students)} ({passing/len(students)*100:.0f}%)")
print(f"Fail: {failing}/{len(students)} ({failing/len(students)*100:.0f}%)")

TOP 3 STUDENTS
  1. Hassan Ali           avg=95.00  grade=A
  2. Tariq Mahmood        avg=92.40  grade=A
  3. Omar Farooq          avg=92.00  grade=A

BOTTOM 3 STUDENTS
  1. Kamran Rauf          avg=40.60  grade=F
  2. Usman Sheikh         avg=50.40  grade=F
  3. Mariam Syed          avg=55.00  grade=F

Pass: 16/20 (80%)
Fail: 4/20 (20%)


## Summary

| Concept | Key point |
|---|---|
| `if/elif/else` | Only ONE branch runs - first match wins |
| `for` over dict | Default: iterates keys. Use `.items()` for key-value pairs |
| `range(start,stop,step)` | Stop is exclusive. `range(5)` = 0,1,2,3,4 |
| `enumerate()` | Always prefer over `range(len(x))` for indexed iteration |
| `zip()` | Stops at shortest iterable - check lengths first if data could be mismatched |
| `break` | Exits immediately - critical for performance in search tasks |
| `continue` | Skips rest of current iteration - good for filtering with early guard clauses |
| Loop variable after loop | Loop variable retains its last value - track explicitly if you need the result |
| `for...else` | `else` block runs only if `break` was never triggered |

**Before Sunday:** Can you explain in words what `for i, (name, score) in enumerate(zip(names, scores)):` does? This is a real pattern you'll see constantly.

In [22]:
students = ["Ali", "Sara", "Ahmed"]

roll_numbers = [101, 102, 103]

marks = [88, 95, 79]

attendance = [92, 85, 98]

for roll_no, name, mark, att in zip(
    roll_numbers,
    students,
    marks,
    attendance
):
    print(
        f"{roll_no} | {name} | Marks: {mark} | Attendance: {att}%"
    )

101 | Ali | Marks: 88 | Attendance: 92%
102 | Sara | Marks: 95 | Attendance: 85%
103 | Ahmed | Marks: 79 | Attendance: 98%
